In [1]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
import numpy as np

In [2]:
cols = ['bedrooms', 'beds', 'baths', 'luxury_items', 'luxury_score', 'location_desirability']

In [3]:
df = pd.read_parquet('./dataset/forCatboost.parquet')
df = df.loc[:, ['Price'] + cols +['quality', 'city']].astype(np.float64, errors='ignore')
vc = df['city'].value_counts()
vc =  vc[vc >= 14]
cities = vc.index
df = df[df['city'].isin(cities)].reset_index(drop=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1474 entries, 0 to 1473
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Price                  1474 non-null   float64
 1   bedrooms               1474 non-null   float64
 2   beds                   1474 non-null   float64
 3   baths                  1474 non-null   float64
 4   luxury_items           1474 non-null   float64
 5   luxury_score           1474 non-null   float64
 6   location_desirability  1474 non-null   float64
 7   quality                1474 non-null   str    
 8   city                   1474 non-null   str    
dtypes: float64(7), str(2)
memory usage: 123.6 KB


In [4]:
X = df.iloc[:, 1:]
y = df['Price']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
model = CatBoostRegressor(cat_features=['city', 'quality'])
model.fit(X_train, y_train)


Learning rate set to 0.042022
0:	learn: 76342.8315174	total: 42.1ms	remaining: 42s
1:	learn: 75017.9284233	total: 83.4ms	remaining: 41.6s
2:	learn: 73779.7060929	total: 124ms	remaining: 41s
3:	learn: 72591.9576394	total: 163ms	remaining: 40.6s
4:	learn: 71557.0951906	total: 199ms	remaining: 39.6s
5:	learn: 70480.7338668	total: 241ms	remaining: 39.9s
6:	learn: 69510.0651231	total: 283ms	remaining: 40.2s
7:	learn: 68534.2291368	total: 324ms	remaining: 40.2s
8:	learn: 67573.8842935	total: 362ms	remaining: 39.8s
9:	learn: 66675.7813454	total: 399ms	remaining: 39.5s
10:	learn: 65815.0174672	total: 434ms	remaining: 39s
11:	learn: 64883.0630951	total: 474ms	remaining: 39s
12:	learn: 64164.9362732	total: 519ms	remaining: 39.4s
13:	learn: 63481.1566561	total: 548ms	remaining: 38.6s
14:	learn: 62760.7196370	total: 600ms	remaining: 39.4s
15:	learn: 62045.5555707	total: 658ms	remaining: 40.5s
16:	learn: 61414.0729913	total: 698ms	remaining: 40.4s
17:	learn: 60889.2265369	total: 745ms	remaining: 40

CatBoostRegressor(cat_features=['city', 'quality'], loss_function='RMSE')

In [43]:
ypred = model.predict(X_test)

print(f'MAE: {mean_absolute_error(y_test, ypred)}')
print(f'MAPE: {mean_absolute_percentage_error(y_test, ypred)}')

MAE: 25484.93353834256
MAPE: 0.3042464253238613


In [71]:
model_final = CatBoostRegressor(cat_features=['city', 'quality'])
model_final.fit(X, y)

Learning rate set to 0.043531
0:	learn: 76402.7165393	total: 39.4ms	remaining: 39.3s
1:	learn: 75202.4156409	total: 81ms	remaining: 40.4s
2:	learn: 73812.2154323	total: 122ms	remaining: 40.5s
3:	learn: 72575.2081877	total: 161ms	remaining: 40.1s
4:	learn: 71466.7426619	total: 201ms	remaining: 40s
5:	learn: 70396.2144188	total: 243ms	remaining: 40.2s
6:	learn: 69405.5180603	total: 282ms	remaining: 40s
7:	learn: 68400.4390205	total: 315ms	remaining: 39s
8:	learn: 67392.4748290	total: 354ms	remaining: 39s
9:	learn: 66461.1522928	total: 393ms	remaining: 38.9s
10:	learn: 65498.3468566	total: 429ms	remaining: 38.6s
11:	learn: 64565.1335791	total: 469ms	remaining: 38.6s
12:	learn: 63776.6971888	total: 508ms	remaining: 38.5s
13:	learn: 63081.6683753	total: 546ms	remaining: 38.5s
14:	learn: 62382.7074382	total: 583ms	remaining: 38.3s
15:	learn: 61595.0136463	total: 618ms	remaining: 38s
16:	learn: 60941.7847988	total: 656ms	remaining: 37.9s
17:	learn: 60395.4138322	total: 737ms	remaining: 40.2s


CatBoostRegressor(cat_features=['city', 'quality'], loss_function='RMSE')

<h1>Optimistic Evaluation</h1>

In [72]:
ypred = model_final.predict(X)

print(f'MAE: {mean_absolute_error(y, ypred)}')
print(f'MAPE: {mean_absolute_percentage_error(y, ypred)}')

MAE: 19355.708541373882
MAPE: 0.299645680922298


In [73]:
model_final.save_model('./GUI/airbnb_catboost.cmb')

In [7]:
X_test.to_csv('test.csv', index=False)